In [ ]:
REPO_URL = "https://github.com/huyvanzzz/finetune-InternVL2.git"
TARGET_BRANCH = "feature/trajectory-pretrain-qformer-concat-bestshot-bf16"
PROJECT_DIR = "/workspace/finetune-InternVL2"
CONFIG_PATH = "internvl_config_traj_concat_bestshot_3frame_bf16_2gpu.yaml"


In [ ]:
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "fetch", "origin", TARGET_BRANCH], check=True)
subprocess.run(["git", "checkout", "-B", TARGET_BRANCH, f"origin/{TARGET_BRANCH}"], check=True)
print("Current repo:", os.getcwd())
print("Current branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Current commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
!python -m py_compile train.py wad_dataset.py qformer_bridge.py scripts/test_infer.py scripts/prepare_qformer.py scripts/smoke_qformer_bridge.py
!python -m pytest tests/test_finetune_alter_only.py -q


In [ ]:
import subprocess
subprocess.run(["python", "build_frame_index.py"], input="n\n", text=True, check=True)


In [ ]:
!python scripts/prepare_qformer.py --config {CONFIG_PATH}
!python scripts/smoke_qformer_bridge.py --config {CONFIG_PATH}


In [ ]:
import subprocess
TRAIN_CHECKPOINT = ""
PRETRAIN_CHECKPOINT = ""
TRAIN_START_EPOCH = None
TRAIN_START_STEP = None
cmd = ["accelerate", "launch", "--num_processes", "2", "train.py", "--config", CONFIG_PATH]
if TRAIN_CHECKPOINT:
    cmd += ["--checkpoint", TRAIN_CHECKPOINT]
if PRETRAIN_CHECKPOINT:
    cmd += ["--pretrain_checkpoint", PRETRAIN_CHECKPOINT]
if TRAIN_START_EPOCH is not None:
    cmd += ["--start_epoch", str(TRAIN_START_EPOCH)]
if TRAIN_START_STEP is not None:
    cmd += ["--start_step", str(TRAIN_START_STEP)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import glob, os, subprocess
RUN_ID = ""
OUTPUT_ROOT = "outputs/internvl3_2b_traj_concat_bestshot_3frame_bf16_2gpu"
if not RUN_ID:
    runs = sorted(glob.glob(os.path.join(OUTPUT_ROOT, "20*")))
    RUN_ID = os.path.basename(runs[-1])
EVAL_ALL_EPOCHS = True
epochs = sorted(glob.glob(os.path.join(OUTPUT_ROOT, RUN_ID, "epoch_*"))) if EVAL_ALL_EPOCHS else [os.path.join(OUTPUT_ROOT, RUN_ID, "epoch_1")]
for checkpoint in epochs:
    epoch_name = os.path.basename(checkpoint)
    output_file = f"results/concat_bestshot_3frame_{RUN_ID}_{epoch_name}_test_alter.json"
    cmd = ["python", "scripts/test_infer.py", "--config", CONFIG_PATH, "--checkpoint", checkpoint, "--split", "test_alter", "--output_file", output_file]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print("Pairs JSON:", output_file.replace(".json", "_pairs.json"))
